# Explicabilidad y simulador de riesgo por vano

Hermano de `01.4_uiti_vano_trayectorias_vano.ipynb`: reutiliza su misma clasificacion
KMeans por vano x ventana (nunca se reajusta aca) y agrega, sobre esa base, un
simulador interactivo de "que pasaria si" a nivel de vano usando el modelo MGCECDL.

**Requiere un kernel Python vivo.** Este cuaderno NO es exportable a HTML estatico ni
publicable como Databricks App, a diferencia de 01.2/01.3/01.4: sus controles
(`ipywidgets`) y el simulador corren en el kernel, no en el navegador.

**Que mide cada mapa (no confundir).** El mapa de la fila 1 (izquierda) es el
**grupo historico**: la clase KMeans que 01.4 ya calculo sobre eventos observados. El
mapa de la fila 2 (derecha) muestra la **clase predicha** por el modelo MGCECDL, con y
sin las variables simuladas -- vease el panel del simulador y el alternador
base/simulado/delta. Son dos mediciones distintas y nunca comparten leyenda ni titulo.

**Estado actual (tras PR5)**: la figura completa de 2x3 paneles con las 14 trazas
esta funcionando de punta a punta -- mapa historico (fila 1), ranking de sensibilidad
min-max (fila 1 col 2), y mapa predicho + simulador con el boton "Simular" y el
alternador base/simulado/delta (fila 2). Solo la columna 3 de la fila 1 (grafo
reconstruido, decision D4) sigue diferida a un PR futuro -- su traza existe, esta
indexada, y se muestra oculta con una anotacion explicita.

In [ ]:
import asyncio
import sys
import time
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

try:
    import ipywidgets as widgets
except ImportError as exc:
    raise ImportError('Este cuaderno requiere ipywidgets para la interfaz interactiva.') from exc
from IPython.display import display

# Sube desde el cwd hasta la raiz del repo (marcada por la carpeta src/), igual que 09.
# Se agregan ROOT y ROOT/src -- no solo src/ -- porque ventanas_015.py importa
# `scripts.extract_geometrias_014` (paquete de nivel de repo, igual que en notebook 10).
ROOT = Path.cwd().resolve()
while not (ROOT / 'src').is_dir() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
for _path_a_agregar in (ROOT, ROOT / 'src'):
    if str(_path_a_agregar) not in sys.path:
        sys.path.insert(0, str(_path_a_agregar))

from chec_impacto.data import preparar_splits_estratificados, procesar_dataset_completo
from chec_impacto.models.criticality_assignment import (
    CLAVE_ESPACIO_CANONICO,
    GEOMETRIAS_SHA1_ESPERADO,
    verificar_sha1_geometrias,
)
from chec_impacto.training import (
    cargar_modelo_mgcecdl,
    escalar_features_minmax_mgcecdl,
    predict_classification,
    resolve_training_device,
)
from chec_local_interpreter.vano_controls import build_knobs, expand_knob_overrides
from chec_local_interpreter.ventanas_015 import (
    capas_mapa_historico,
    construir_hist_class_cache,
    construir_mask_cache,
    construir_tabla_vano_ventana,
    construir_ventanas,
)
from scripts.extract_geometrias_014 import (
    DEFAULT_NOTEBOOK_PATH,
    DEFAULT_OUTPUT_PATH,
    extraer_geometrias_014,
)

In [ ]:
# Ventana climatica igual que 03_mgcecdl_training / 09_simulador: cambiarla generaria un
# set de features distinto al que el modelo cargado en la celda SEAM espera.
VENTANA_CLIMATICA_HORAS = 12
CLAVE_ESPACIO = CLAVE_ESPACIO_CANONICO  # '2' -- espacio canonico fijado en criticality_assignment.py
DEVICE = resolve_training_device('auto')

# Misma paleta que 01.4: los grupos historicos de este cuaderno SON los de 01.4, nunca se
# reajustan, asi que el color tiene que significar lo mismo en los dos cuadernos.
NOMBRES_GRUPOS = ['Bajo', 'Medio', 'Medio-Alto', 'Alto']
COLORES_GRUPOS = ['rgb(252,187,161)', 'rgb(251,106,74)', 'rgb(203,24,29)', 'rgb(103,0,13)']
COLOR_SIN_DATO = '#94a3b8'  # mismo gris "sin grupo" de 01.3/01.4, distinto de los 4 colores
COLOR_AUN_NO_SIMULADO = '#a78bfa'  # violeta, distinto de COLOR_SIN_DATO: fila 2 antes de
# la primera simulacion no comparte color/leyenda con "simulado, sin filas de evento" (D2, hallazgo W1)
COLOR_MARCADO = '#0072b2'
ANCHO_MAPA = 3.0
ANCHO_MAPA_MARCADO = round(ANCHO_MAPA * 1.4, 2)

In [ ]:
# --- Reutilizacion de la geometria KMeans de 01.4 (design section F) -------
# Falla RAPIDO aca, antes de procesar el dataset completo (celda siguiente): si 01.4 fue
# editado y sus centroides se movieron, no tiene sentido esperar el procesamiento pesado
# para enterarse. `cargar_clases_desde_014` (celda 7, via hist_class_cache) repite esta
# misma verificacion por cada ventana consultada -- barata, y evita que una geometria
# cacheada quede sin recomprobar dentro de la misma sesion.
GEOMETRIAS_PATH = DEFAULT_OUTPUT_PATH
if not GEOMETRIAS_PATH.exists():
    extraer_geometrias_014(DEFAULT_NOTEBOOK_PATH, GEOMETRIAS_PATH)
_sha1_real, _coincide = verificar_sha1_geometrias(GEOMETRIAS_PATH, esperado=GEOMETRIAS_SHA1_ESPERADO)
assert _coincide, (
    f'La geometria KMeans extraida de 01.4 no coincide con la esperada '
    f'(esperado={GEOMETRIAS_SHA1_ESPERADO}, real={_sha1_real}). 01.4 fue modificado; '
    f'01.5 depende de esa geometria.'
)
print(f'Geometria 01.4 verificada -- sha1 coincide ({_sha1_real[:12]}...)')

In [ ]:
DATA_PATH = ROOT / 'data' / 'Indicadores_vano_v3.csv'
VARIABLES_SELECCION_PATH = ROOT / 'data' / 'Variables_seleccion.xlsx'
MODEL_DIR = ROOT / 'data' / 'models'

# Mismo preprocesamiento real usado en entrenamiento (03_mgcecdl_training / 09_simulador):
# sin muestreo ni filtro de UITI, para que context_df quede alineado FILA A FILA con X --
# la clave que permite reusar la MISMA mascara (circuito, ventana) para el mapa historico
# (sin modelo) y, en un PR futuro, para las predicciones del modelo sobre esas mismas filas.
datos = procesar_dataset_completo(
    path_clima=DATA_PATH,
    path_variables_seleccion=VARIABLES_SELECCION_PATH,
    use_sampling=False,
    min_samples_per_codigo=5,
    target='UITI_VANO',
    filtro_uiti_max=None,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

feature_names = list(datos['features'])
X_raw_model = np.asarray(datos['X'], dtype=np.float32)
Xdf = datos['Xdata'].copy().reset_index(drop=True)
context_df = datos['df_original_copy'].copy().reset_index(drop=True)
label_encoders = datos.get('label_encoders', {})
max_values_imputed = datos.get('max_values_imputed', {})

splits_clf = escalar_features_minmax_mgcecdl(
    preparar_splits_estratificados(
        X_raw_model, datos['y'], modo='clasificacion', random_state=42,
    )
)
feature_scaler = splits_clf['feature_scaler']
X = feature_scaler.transform(X_raw_model).astype(np.float32)

assert len(context_df) == len(X), 'context_df y X deben quedar alineados fila a fila'
print(f'{len(context_df):,} filas | {len(feature_names)} features | X{X.shape}')

In [ ]:
# --- SEAM D1: cambiar SOLO este bloque para pasar a MIL (design section D) -------------
MODEL_PATH = MODEL_DIR / 'mgcecdl_classifier_best.zip'  # nombre FIJO, no "el mas reciente":
# evita levantar por error un checkpoint todavia en curso de otra sesion en este repo.
MODELO = cargar_modelo_mgcecdl(str(MODEL_PATH), device=DEVICE)
PREDICT_FN = predict_classification
# MIL (PR futuro):
# MODELO = BagPredictor(mil_model, feature_names=FEATURES_MIL, geometria=GEOMETRIA)
# PREDICT_FN = mil_vano_ventana.predict_fn

_probe = PREDICT_FN(MODELO, X[: min(len(X), 8)], device=DEVICE, batch_size=8)
assert set(_probe) >= {'fused_probs', 'predicted_classes'}
N_CLASSES = int(np.asarray(_probe['fused_probs']).shape[1])
print(f'Modelo cargado -- {N_CLASSES} clases (contrato PREDICT_FN verificado)')

In [ ]:
# --- construir_ventanas + per-(vano, ventana) events + caches (design section A) -------
VENTANAS = construir_ventanas(context_df['FECHA'])
TABLA = construir_tabla_vano_ventana(context_df, VENTANAS)
mask_para = construir_mask_cache(TABLA)
clases_para = construir_hist_class_cache(TABLA, mask_para)

CIRCUITOS = sorted(TABLA['CIRCUITO'].astype(str).unique())
VANOS_POR_CIRCUITO = {
    c: sorted(g['FID_VANO'].unique().tolist())
    for c, g in TABLA.groupby(TABLA['CIRCUITO'].astype(str))
}

print(f'{len(TABLA):,} celdas vano x ventana con eventos | {len(VENTANAS)} ventanas | '
      f'{TABLA["FID_VANO"].nunique():,} vanos distintos | {len(CIRCUITOS)} circuitos')


# Geometria FISICA de cada vano (no confundir con la geometria KMeans de la celda 4): mismo
# shapefile y mismo join que el mapa de 01.3/01.4. No se extrae a src/ porque es solo
# lectura + reindexado geoespacial, sin logica propia que valga la pena testear por fuera
# de lo que TABLA/capas_mapa_historico ya cubren.
def _norm_id(serie):
    return (serie.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .replace({'': pd.NA, '<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA}))


_lineas = gpd.read_file(ROOT / 'data' / 'GEO' / 'MVLINSEC.shp')
if str(_lineas.crs) != 'EPSG:4326':
    _lineas = _lineas.to_crs('EPSG:4326')
_lineas['FID_VANO_GEO'] = _norm_id(_lineas['G3E_FID'])
_utiles = _lineas[_lineas['CIRCUITO'].astype(str).isin(set(CIRCUITOS))]

GEO_POR_CIRCUITO = {}
for _c, _g in _utiles.groupby(_utiles['CIRCUITO'].astype(str)):
    fids, lats, lons = [], [], []
    for _fid, _geom in zip(_g['FID_VANO_GEO'], _g.geometry):
        if _geom is None or _geom.is_empty:
            continue
        for _p in ([_geom] if _geom.geom_type == 'LineString' else list(getattr(_geom, 'geoms', []))):
            xs, ys = _p.xy
            fids.append(str(_fid))
            lats.append([round(v, 5) for v in ys])
            lons.append([round(v, 5) for v in xs])
    if fids:
        GEO_POR_CIRCUITO[_c] = {'fids': fids, 'lat': lats, 'lon': lons}

print(f'{len(GEO_POR_CIRCUITO)} circuitos con geometria fisica')

In [ ]:
KNOBS = build_knobs(
    feature_names=feature_names,
    original_feature_df=Xdf,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)
print(f'{len(KNOBS)} controles (Knob catalog, PR2a) -- '
      f'{sum(1 for k in KNOBS if k.kind == "categorical")} categoricos, '
      f'{sum(1 for k in KNOBS if k.kind == "numeric")} numericos, '
      f'{sum(1 for k in KNOBS if k.kind == "constant")} constantes')

In [ ]:
# --- Inventario de trazas CONGELADO (design section G). Los indices 0-13 son fijos: la
# columna 3 de la fila 1 (grafo reconstruido, decision D4) muta el indice 7 en un PR
# futuro, y ese mismo PR agrega trazas nuevas a partir del indice 14 -- ningun indice
# EXISTENTE se mueve nunca. La grilla es 2x3 y se queda 2x3.
IDX = {
    'clases': [0, 1, 2, 3],          # fila 1 col 1 -- mapa historico (01.4), PR3
    'sin_dato': 4,                    # fila 1 col 1 -- sin eventos en la ventana
    'marcados': 5,                    # fila 1 col 1 -- halo de vanos marcados
    'ranking': 6,                     # fila 1 col 2 -- sensibilidad min-max, PR4
    'diferido': 7,                    # fila 1 col 3 -- RESERVADO, decision D4
    'pred_clases': [8, 9, 10, 11],    # fila 2 col 1 -- mapa predicho MGCECDL, PR5
    'pred_sin_dato': 12,               # fila 2 col 1, PR5
    'pred_marcados': 13,               # fila 2 col 1, PR5
}

_fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type': 'map'}, {'type': 'xy'}, {'type': 'xy'}],
           [{'type': 'map'}, {'type': 'xy'}, {'type': 'xy'}]],
    subplot_titles=(
        'Grupo historico (01.4) -- clase por vano en la ventana (eventos observados)',
        'Sensibilidad min-max -- vanos marcados',
        '',
        'Clase predicha MGCECDL -- modelo (base/simulado/delta, ver nota)',
        '',
        '',
    ),
    horizontal_spacing=0.08, vertical_spacing=0.16,
)

for _clase in range(4):                                          # 0-3
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='hist', legendgrouptitle_text='Grupo historico',
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 4
    lat=[], lon=[], mode='lines', name='Sin dato', legendgroup='hist',
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_DATO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)
_fig.add_trace(go.Scattermap(                                    # 5
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='hist',
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_MARCADO),
    hovertext=[], hoverinfo='text',
), row=1, col=1)

_fig.add_trace(go.Bar(x=[], y=[], orientation='h', showlegend=False), row=1, col=2)  # 6

_fig.add_trace(go.Scatter(x=[], y=[], mode='markers', visible=False, showlegend=False), row=1, col=3)  # 7
_fig.update_xaxes(visible=False, showticklabels=False, row=1, col=3)
_fig.update_yaxes(visible=False, showticklabels=False, row=1, col=3)
_fig.add_annotation(
    text='Panel del grafo reconstruido<br><sup>diferido a un PR futuro (decision D4)</sup>',
    xref='x2 domain', yref='y2 domain', x=0.5, y=0.5, showarrow=False,
    font=dict(size=12, color='#7a5c58'),
)

for _clase in range(4):                                          # 8-11
    _fig.add_trace(go.Scattermap(
        lat=[], lon=[], mode='lines', name=NOMBRES_GRUPOS[_clase],
        legendgroup='pred', legendgrouptitle_text='Clase predicha', showlegend=False,
        line=dict(width=ANCHO_MAPA, color=COLORES_GRUPOS[_clase]),
        hovertext=[], hoverinfo='text',
    ), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 12
    lat=[], lon=[], mode='lines', name='Sin dato', legendgroup='pred', showlegend=False,
    line=dict(width=ANCHO_MAPA, color=COLOR_SIN_DATO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)
_fig.add_trace(go.Scattermap(                                    # 13
    lat=[], lon=[], mode='lines', name='Vano marcado', legendgroup='pred', showlegend=False,
    line=dict(width=ANCHO_MAPA_MARCADO, color=COLOR_MARCADO),
    hovertext=[], hoverinfo='text',
), row=2, col=1)

_fig.update_layout(
    map=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    map2=dict(style='carto-positron', center=dict(lat=5.07, lon=-75.52), zoom=10),
    title=dict(text='01.5 -- Explicabilidad y simulador de riesgo por vano'),
    height=760, width=1280, template='plotly_white',
)

# Los indices se verifican al generar, igual que 01.3/01.4: si alguien reordena las
# trazas esto falla ACA, no se descubre silenciosamente en la celda de dibujo.
assert len(_fig.data) == 14, len(_fig.data)
assert all(_fig.data[i].type == 'scattermap' for i in IDX['clases'] + [IDX['sin_dato'], IDX['marcados']])
assert [_fig.data[i].line.color for i in IDX['clases']] == COLORES_GRUPOS
assert _fig.data[IDX['ranking']].type == 'bar'
assert _fig.data[IDX['diferido']].type == 'scatter' and _fig.data[IDX['diferido']].visible is False
assert all(_fig.data[i].type == 'scattermap'
           for i in IDX['pred_clases'] + [IDX['pred_sin_dato'], IDX['pred_marcados']])

fig = go.FigureWidget(_fig)
print(f'FigureWidget con {len(fig.data)} trazas (indices 0-13 congelados, design section G)')

In [ ]:
def _seleccion_actual():
    return circuito_widget.value, ventana_widget.value, set(vano_widget.value)


def _redibujar_mapa_historico(*_ignorado):
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    clases_por_fid = clases_para(circuito, ventana_i)
    capas = capas_mapa_historico(geo, clases_por_fid, marcados=marcados)
    with fig.batch_update():
        for _clase in range(4):
            fig.data[IDX['clases'][_clase]].lat = capas['clases'][_clase]['lat']
            fig.data[IDX['clases'][_clase]].lon = capas['clases'][_clase]['lon']
        fig.data[IDX['sin_dato']].lat = capas['sin_dato']['lat']
        fig.data[IDX['sin_dato']].lon = capas['sin_dato']['lon']
        fig.data[IDX['marcados']].lat = capas['marcados']['lat']
        fig.data[IDX['marcados']].lon = capas['marcados']['lon']


circuito_widget = widgets.Dropdown(options=CIRCUITOS, description='Circuito')
ventana_widget = widgets.SelectionSlider(
    options=[(v['etiqueta'], v['i']) for v in VENTANAS],
    description='Ventana', continuous_update=False,
)
vano_widget = widgets.SelectMultiple(
    options=VANOS_POR_CIRCUITO.get(circuito_widget.value, []), description='Vanos', rows=6,
)


def _on_circuito_change(_change):
    vano_widget.options = VANOS_POR_CIRCUITO.get(circuito_widget.value, [])
    vano_widget.value = ()
    _redibujar_mapa_historico()


# Tier 0 del presupuesto de interactividad (design section A): elegir circuito, mover la
# ventana o marcar un vano no llama al modelo -- sin debounce ni epoch guard, que
# pertenecen al tier 1/2 (fila 2, ranking, boton "Simular"), fuera del alcance de este PR.
circuito_widget.observe(_on_circuito_change, names='value')
ventana_widget.observe(_redibujar_mapa_historico, names='value')
vano_widget.observe(_redibujar_mapa_historico, names='value')

_redibujar_mapa_historico()  # primer dibujo, con la seleccion inicial

APP = widgets.HBox([widgets.VBox([circuito_widget, ventana_widget, vano_widget]), fig])

In [ ]:
# --- Fila 2: mapa predicho MGCECDL + alternador base/simulado/delta + boton "Simular" --
# (design section A, decision D2, spec "Row 2 anti-conflation and base/simulado/delta
# toggle"). El alternador NO llama al modelo de nuevo: las 3 vistas salen del MISMO
# resultado de `simulate_explicit_overrides` (design section C) -- cero pasadas extra.
# "Simular" SI llama al modelo -- exactamente 2 pasadas (base + simulado), tier 1 del
# presupuesto de interactividad (<=1.5 s). Debounce asincronico (design section A):
# `asyncio.ensure_future` + cancelacion en el propio event loop del kernel, NUNCA
# `threading.Timer` -- ipykernel enruta la salida de los widgets con el parent header
# thread-local, asi que una escritura desde un hilo en segundo plano cae en la celda
# equivocada. `_EPOCA` es el guard de epoca COMPARTIDO con la celda de ranking (tier 2,
# siguiente celda): cualquier evento que invalide un job en vuelo -- de cualquier tier --
# avanza la MISMA epoca.
from chec_local_interpreter.simulator import simulate_explicit_overrides
from chec_local_interpreter.vano_app_015 import (
    ALTERNADOR_POR_DEFECTO,
    DEBOUNCE_SEGUNDOS,
    ESTADO_POR_ETIQUETA,
    ETIQUETAS_ALTERNADOR,
    ETIQUETAS_DELTA,
    aplicar_si_vigente,
    clases_por_fid_para_estado,
    construir_evento_mask_cache,
    etiqueta_capa_sin_dato,
    siguiente_epoca,
    submuestrear_si_excede,
)
from chec_local_interpreter.vano_widgets import widget_for_knob

mask_evento_para = construir_evento_mask_cache(context_df, VENTANAS)

_EPOCA = 0
_tarea_pendiente_simular = None
_ultimo_resultado_simulacion = None   # DataFrame de simulate_explicit_overrides, o None
_ultima_seleccion_simulada = None     # (circuito, ventana_i) al que corresponde ese resultado

STATUS = widgets.HTML(
    'Fila 2 sin simular todavia -- elige variables (opcional) y presiona "Simular".'
)

NOTA_ANTICONFUSION = widgets.HTML(
    '<div style="font-size:0.85em;color:#7a5c58;border:1px solid #e5c7c3;'
    'border-radius:4px;padding:6px 8px;max-width:420px;">'
    '<b>No confundir los dos mapas.</b> El de la fila 1 es el <b>grupo historico</b> '
    '(01.4, sobre eventos observados). El de la fila 2 es la <b>clase predicha</b> por '
    'el modelo MGCECDL, con o sin variables simuladas. Cualquier diferencia entre ambos '
    'mezcla el efecto de la simulacion con la brecha modelo-vs-historia -- no son la '
    'misma medicion.</div>'
)

_knobs_por_id = {k.id: k for k in KNOBS}
knob_selector_widget = widgets.SelectMultiple(
    options=[(k.label, k.id) for k in KNOBS], description='Variables', rows=6,
)
controles_knob_box = widgets.VBox([])
_controles_knob_actuales = {}


def _reconstruir_controles_knob(_change=None):
    global _controles_knob_actuales
    _controles_knob_actuales = {
        knob_id: widget_for_knob(_knobs_por_id[knob_id]) for knob_id in knob_selector_widget.value
    }
    controles_knob_box.children = list(_controles_knob_actuales.values())


knob_selector_widget.observe(_reconstruir_controles_knob, names='value')

alternador_widget = widgets.ToggleButtons(
    options=list(ETIQUETAS_ALTERNADOR), value=ALTERNADOR_POR_DEFECTO, description='Vista',
)
boton_simular = widgets.Button(description='Simular', button_style='primary')


def _redibujar_mapa_predicho(*_ignorado):
    """Repaint puro, CERO llamadas al modelo: usa `_ultimo_resultado_simulacion` (o
    nada, si todavia no hay simulacion para la seleccion activa) segun el estado
    actual del alternador."""
    circuito, ventana_i, marcados = _seleccion_actual()
    geo = GEO_POR_CIRCUITO.get(circuito, {'fids': [], 'lat': [], 'lon': []})
    hay_resultado = (
        _ultimo_resultado_simulacion is not None
        and _ultima_seleccion_simulada == (circuito, ventana_i)
    )
    if hay_resultado:
        estado = ESTADO_POR_ETIQUETA[alternador_widget.value]
        clases_por_fid = clases_por_fid_para_estado(_ultimo_resultado_simulacion, estado)
    else:
        clases_por_fid = {}
    capas = capas_mapa_historico(geo, clases_por_fid, marcados=marcados)
    # Fila 2 anti-conflation, hallazgo W1: "aun no simulado" (hay_resultado=False, TODOS los
    # vanos en sin_dato) y "simulado, sin filas de evento" (hay_resultado=True, este vano en
    # particular en sin_dato) son dos razones distintas para la misma casilla -- se distinguen
    # con color Y leyenda propios, no solo en el texto de STATUS.
    etiqueta_sin_dato = etiqueta_capa_sin_dato(hay_resultado)
    color_sin_dato = COLOR_SIN_DATO if hay_resultado else COLOR_AUN_NO_SIMULADO

    es_delta = alternador_widget.value == 'Δ simulado − base'
    etiquetas_clase = list(ETIQUETAS_DELTA) if es_delta else NOMBRES_GRUPOS
    titulo_grupo = 'Δ simulado − base' if es_delta else 'Clase predicha'
    with fig.batch_update():
        for _clase in range(4):
            fig.data[IDX['pred_clases'][_clase]].lat = capas['clases'][_clase]['lat']
            fig.data[IDX['pred_clases'][_clase]].lon = capas['clases'][_clase]['lon']
            fig.data[IDX['pred_clases'][_clase]].name = etiquetas_clase[_clase]
            fig.data[IDX['pred_clases'][_clase]].legendgrouptitle = dict(text=titulo_grupo)
            fig.data[IDX['pred_clases'][_clase]].showlegend = True
        fig.data[IDX['pred_sin_dato']].lat = capas['sin_dato']['lat']
        fig.data[IDX['pred_sin_dato']].lon = capas['sin_dato']['lon']
        fig.data[IDX['pred_sin_dato']].name = etiqueta_sin_dato
        fig.data[IDX['pred_sin_dato']].line.color = color_sin_dato
        fig.data[IDX['pred_sin_dato']].showlegend = True
        fig.data[IDX['pred_marcados']].lat = capas['marcados']['lat']
        fig.data[IDX['pred_marcados']].lon = capas['marcados']['lon']
        fig.data[IDX['pred_marcados']].showlegend = True


def _limpiar_resultado_simulacion(_change=None):
    """Circuito o ventana cambiaron: el ultimo resultado ya NO corresponde a la
    seleccion activa -- se descarta (fila 2 vuelve a "sin dato") en vez de mostrar la
    prediccion de OTRA seleccion, que violaria la regla anti-confusion (D2)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada, _EPOCA
    _ultimo_resultado_simulacion = None
    _ultima_seleccion_simulada = None
    _EPOCA = siguiente_epoca(_EPOCA)  # invalida cualquier job tier1/tier2 en vuelo
    _redibujar_mapa_predicho()


def _simular(epoca_job):
    """Computo pesado tier 1 -- bloqueante dentro de la corutina (design section A: un
    job ya iniciado no se puede interrumpir). Guarda el resultado y repinta SOLO si
    `epoca_job` sigue vigente al terminar (epoch guard)."""
    global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
    circuito, ventana_i, _marcados = _seleccion_actual()
    mask = mask_evento_para(circuito, ventana_i)
    if not mask.any():
        aplicar_si_vigente(
            lambda: setattr(STATUS, 'value', 'Sin filas de evento para esta seleccion.'),
            epoca_job=epoca_job, epoca_actual=lambda: _EPOCA,
        )
        return

    mask_efectiva, submuestreado, n_total = submuestrear_si_excede(mask)
    overrides = expand_knob_overrides(
        {knob_id: widget.value for knob_id, widget in _controles_knob_actuales.items()}, KNOBS,
    )

    t0 = time.perf_counter()
    resultado, metadata = simulate_explicit_overrides(
        model=MODELO, X_scaled=X, X_raw_model=X_raw_model, feature_names=feature_names,
        feature_scaler=feature_scaler, predict_fn=PREDICT_FN, device=DEVICE,
        mask=mask_efectiva, vano_ids=context_df['FID_VANO'], overrides=overrides,
        label_encoders=label_encoders, max_values_imputed=max_values_imputed,
    )
    duracion = time.perf_counter() - t0

    def _escribir():
        global _ultimo_resultado_simulacion, _ultima_seleccion_simulada
        _ultimo_resultado_simulacion = resultado
        _ultima_seleccion_simulada = (circuito, ventana_i)
        muestra = f' (muestra de {n_total:,} filas)' if submuestreado else ''
        STATUS.value = (
            f'{duracion:.2f} s | {metadata["n_vanos"]} vanos | {metadata["n_registros"]:,} filas'
            f'{muestra} | {len(overrides)} variables aplicadas'
        )
        _redibujar_mapa_predicho()

    aplicar_si_vigente(_escribir, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_simulacion(*_ignorado):
    global _EPOCA, _tarea_pendiente_simular
    if _tarea_pendiente_simular is not None and not _tarea_pendiente_simular.done():
        _tarea_pendiente_simular.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA
    STATUS.value = 'Simulando...'

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _simular(epoca_job)

    _tarea_pendiente_simular = asyncio.ensure_future(_tarea())


boton_simular.on_click(_programar_simulacion)
alternador_widget.observe(lambda _c: _redibujar_mapa_predicho(), names='value')
circuito_widget.observe(_limpiar_resultado_simulacion, names='value')
ventana_widget.observe(_limpiar_resultado_simulacion, names='value')
vano_widget.observe(_redibujar_mapa_predicho, names='value')  # solo redibuja el halo marcado

_redibujar_mapa_predicho()  # primer dibujo: sin simulacion todavia -> "sin dato" en toda la fila 2


In [ ]:
# --- Ranking de sensibilidad min-max, fila 1 col 2 (design section A, decision D7) -----
# Grano: vanos MARCADOS en la ventana activa (nunca todo el circuito). Recalcula cuando
# cambia el conjunto marcado o la ventana. Tier 2 del presupuesto de interactividad
# (design section A). PR5 completa el debounce asincronico + epoch guard que faltaban
# aqui (compartidos con el tier 1 del boton "Simular", celda anterior) y agrega la
# segunda escalera de degradacion: si una corrida real supera
# UMBRAL_DEGRADACION_SEGUNDOS, "Relevancias automaticas" se desactiva sola.
from chec_local_interpreter.relevancias_015 import construir_relevance_cache, fingerprint
from chec_local_interpreter.vano_app_015 import UMBRAL_DEGRADACION_SEGUNDOS, debe_degradar_a_manual

FINGERPRINT_ACTUAL = fingerprint(
    geometrias_sha1=GEOMETRIAS_SHA1_ESPERADO,
    model_path=MODEL_PATH,
    feature_names=feature_names,
    ventana_climatica_horas=VENTANA_CLIMATICA_HORAS,
)

rankear_relevancia = construir_relevance_cache(
    model=MODELO,
    X_scaled=X,
    X_raw_model=X_raw_model,
    original_feature_df=Xdf,
    feature_names=feature_names,
    knobs=KNOBS,
    feature_scaler=feature_scaler,
    predict_fn=PREDICT_FN,
    device=DEVICE,
    context_df=context_df,
    ventanas=VENTANAS,
    fingerprint_actual=FINGERPRINT_ACTUAL,
    label_encoders=label_encoders,
    max_values_imputed=max_values_imputed,
)


def _redibujar_ranking(*_ignorado):
    circuito, ventana_i, marcados = _seleccion_actual()
    resultado = rankear_relevancia(circuito, ventana_i, marcados)
    # Orden ascendente: en un bar horizontal Plotly dibuja la primera categoria abajo, asi
    # que el mas relevante (primero en `resultado['filas']`, ya ordenado descendente por
    # `simulate_automatic_minmax_sensitivity`) queda arriba.
    filas = list(reversed(resultado['filas']))
    with fig.batch_update():
        fig.data[IDX['ranking']].y = [fila['label'] for fila in filas]
        fig.data[IDX['ranking']].x = [fila['magnitud_max_cambio_abs'] for fila in filas]
        fig.update_yaxes(title_text=resultado['mensaje'] if resultado['vacio'] else None, row=1, col=2)


auto_ranking_widget = widgets.Checkbox(value=True, description='Relevancias automaticas')
boton_recalcular_ranking = widgets.Button(description='Recalcular relevancias')
_tarea_pendiente_ranking = None


def _redibujar_ranking_con_guard(epoca_job):
    t0 = time.perf_counter()
    _redibujar_ranking()
    duracion = time.perf_counter() - t0

    def _post_escritura():
        if debe_degradar_a_manual(duracion) and auto_ranking_widget.value:
            auto_ranking_widget.value = False
            STATUS.value = (
                f'Relevancias automaticas desactivadas: la ultima corrida tomo {duracion:.1f} s '
                f'(> {UMBRAL_DEGRADACION_SEGUNDOS:.0f} s). Usa "Recalcular relevancias" manualmente.'
            )

    aplicar_si_vigente(_post_escritura, epoca_job=epoca_job, epoca_actual=lambda: _EPOCA)


def _programar_ranking(*_ignorado):
    global _EPOCA, _tarea_pendiente_ranking
    if not auto_ranking_widget.value:
        return
    if _tarea_pendiente_ranking is not None and not _tarea_pendiente_ranking.done():
        _tarea_pendiente_ranking.cancel()
    _EPOCA = siguiente_epoca(_EPOCA)
    epoca_job = _EPOCA

    async def _tarea():
        try:
            await asyncio.sleep(DEBOUNCE_SEGUNDOS)
        except asyncio.CancelledError:
            return
        _redibujar_ranking_con_guard(epoca_job)

    _tarea_pendiente_ranking = asyncio.ensure_future(_tarea())


def _click_recalcular_ranking(_boton):
    global _EPOCA
    _EPOCA = siguiente_epoca(_EPOCA)
    _redibujar_ranking_con_guard(_EPOCA)


boton_recalcular_ranking.on_click(_click_recalcular_ranking)
ventana_widget.observe(_programar_ranking, names='value')
vano_widget.observe(_programar_ranking, names='value')

_redibujar_ranking()  # primer dibujo directo: sin vanos marcados todavia -> estado vacio explicito

APP = widgets.HBox([
    widgets.VBox([
        circuito_widget, ventana_widget, vano_widget,
        NOTA_ANTICONFUSION,
        widgets.HTML('<b>Ranking (sensibilidad min-max)</b>'),
        auto_ranking_widget, boton_recalcular_ranking,
        widgets.HTML('<b>Simulador (fila 2)</b>'),
        knob_selector_widget, controles_knob_box, alternador_widget, boton_simular,
        STATUS,
    ]),
    fig,
])


In [ ]:
display(APP)

## Como leerlo

- **El mapa de la izquierda (fila 1) es el grupo HISTORICO**, calculado por 01.4 sobre
  eventos observados -- nunca sobre una prediccion. **El de la derecha (fila 2) es la
  clase PREDICHA** por el modelo MGCECDL, con o sin las variables simuladas. No
  comparten leyenda ni titulo a proposito: mezclarlos invitaria a leer una prediccion
  como si fuera un hecho observado.
- **"Sin dato"** (gris, un color distinto de las 4 clases en ambos mapas) es un vano sin
  eventos en la ventana elegida -- no es el grupo mas bajo, es la ausencia de dato. En la
  fila 2, antes de presionar "Simular" por primera vez para la seleccion activa, el mapa
  usa un color y una leyenda DISTINTOS -- **"Aun no simulado"** (violeta) -- para no
  confundir "todavia no corri el modelo" con "corri el modelo y este vano no tiene filas
  de evento", que son dos razones distintas para la misma casilla.
- **La columna 3 de la fila 1 esta reservada** para un panel futuro (grafo reconstruido,
  decision D4): la traza existe, esta oculta, y la grilla sigue siendo 2x3 -- no se va a
  reacomodar cuando ese PR la pueble.
- **"Sensibilidad min-max"** (fila 1, columna 2) NO es SHAP: es un barrido min-max sobre
  los vanos MARCADOS en la ventana activa. Se llama asi en todo el cuaderno,
  deliberadamente, para no sugerir una tecnica que no se esta usando.
- **El alternador `Predicho (base)` / `Predicho (simulado)` / `Δ simulado − base`**
  (fila 2) no vuelve a llamar al modelo: las 3 vistas salen del MISMO resultado de
  "Simular" -- cero pasadas extra por cambiar de vista. `Δ simulado − base` esta
  agrupado en 4 rangos (`<=-0.5`, `<0`, `>0`, `>=+0.5`), no es un color continuo.
- **"Simular"** aplica solo las variables elegidas en el panel "Variables": cada una
  aparece como un control -- deslizador para numericas, lista desplegable para
  categoricas -- y una familia climatica (precipitacion, temperatura, rafaga y viento)
  se propaga a sus 12 rezagos horarios de una sola vez. Corre exactamente 2 pasadas del
  modelo (base + simulado) sobre las filas de la seleccion activa, dentro del
  presupuesto de interactividad de <=1.5 s (design section A).
- **Cambiar circuito o ventana descarta la ultima simulacion**: la fila 2 vuelve a "Aun no
  simulado" hasta la proxima vez que se presione "Simular" -- mostrar una prediccion de
  OTRA seleccion violaria la misma regla anti-confusion de arriba.
- **Este cuaderno requiere un kernel Python vivo** -- no es exportable a HTML estatico ni
  publicable como Databricks App.